# Sharpening metacells

See `INSTALL.md` for installing what this needs. Nothing here installs anything: if the cell below
fails, the environment is not set up, and the notebook says so rather than working around it.

In [1]:
import numpy as np

import dafpy as dp
import metacellspy as mc
import somegraphspy as sg

print("dafpy", dp.__version__)
print("somegraphspy", sg.__version__)
print("metacellspy", mc.__version__)

# Read from Julia at import, so printing it means Python reached Julia rather than merely that the
# Python packages are installed.
print("regularization", mc.GENE_FRACTION_REGULARIZATION_FOR_CELLS)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


[ Info: Will cache ispath data forever
[ Info: Old linux kernel, will pre-populate mmap into RAM disk: Linux version 4.18.0-553.30.1.el8_10.x86_64 (mockbuild@x64-builder01.almalinux.org) (gcc version 8.5.0 20210514 (Red Hat 8.5.0-22) (GCC)) #1 SMP Tue Nov 26 02:30:26 EST 2024


dafpy 0.3.0
somegraphspy 0.2.0
metacellspy 0.1.0
regularization 0.0001


## Importing the cells


In [2]:
# What to take out of the `AnnData`, and under what name. Anything not named here is copied as it is,
# after the importer's own renaming: a `something_cell` or `something_gene` mask arrives as
# `is_something`, and a `something_umis` as `something_UMIs`. Naming a property here overrides that
# for it alone, so the rest of the import is unaffected.
COPY_DATA = {
    # Which metacell each cell belongs to. This is what the pipeline sharpens rather than computes,
    # and it is the one property the importer skips by default, since it usually comes from a
    # separate metacells file. Here the cells are the only place it exists, so we ask for it.
    ("cell", "metacell_name"): ("metacell", None),
    # The type of each cell. **Specify this whenever the data has a type per cell**: the type axis is
    # built from a vector called `type`, and the column holding it is rarely called that. Leave it
    # out and everything still runs, with no types and uncolored graphs.
    ("cell", "cell_type"): ("type", None),
    # This dataset has a column of its own called `type` - the platform each cell was measured on,
    # which is not a cell type at all. Left alone it would collide with the line above.
    ("cell", "type"): ("platform", None),
    #
    # The batch, the plate it was on, and the run it was sequenced in. These become axes of their
    # own further down, so they are given the names those axes will have. Three other columns hold
    # the same batch identifier - `batch_set_id` is identical to it, `plate` is it with 1212 cells
    # saying the literal string `NA`, and `Plate` is it with the 10x cells left blank - so they are
    # dropped rather than imported and then explained.
    ("cell", "amp_batch_id"): ("batch", None),
    ("cell", "batch_set_id"): None,
    ("cell", "Plate"): None,
    ("cell", "plate"): None,
    ("cell", "Plate.."): ("plate", None),
    ("cell", "seq_batch_id"): ("sequencing_run", None),
    #
    # The wet lab's record of each batch, plate and run. These are spelled as they were typed into a
    # spreadsheet, dots, capitals, typos and all, and are about to become properties of those axes
    # where they will be read rather than merely stored.
    ("cell", "Comment"): ("comment", None),
    ("cell", "Conc...ng.ul."): ("concentration_ng_per_ul", None),
    ("cell", "Evarage.size..bp."): ("average_size_bp", None),
    ("cell", "External.Index"): ("external_index", None),
    ("cell", "Internal.Index"): ("internal_index", None),
    ("cell", "QC1"): ("qc1", None),
    ("cell", "QC2"): ("qc2", None),
    ("cell", "delta_CT"): ("delta_ct", None),
    ("cell", "Libprep.Cycles"): ("libprep_cycles", None),
    ("cell", "Owner"): ("owner", None),
    ("cell", "Plate.Date"): ("plate_date", None),
    ("cell", "Production.Date"): ("production_date", None),
    ("cell", "Sort.Date"): ("sort_date", None),
    ("cell", "Last.sequensing.date"): ("last_sequencing_date", None),
    ("cell", "Sequencing.Dates"): ("sequencing_dates", None),
    ("cell", "Experiment"): ("experiment", None),
    # Not a genotype: the values are free text describing the sample the batch was made from - the
    # strain, the stage, which embryos, whether it is placenta - and only some of them are strains.
    # It is constant per batch and per plate, and *not* per embryo, which is the giveaway.
    ("cell", "Genotype"): ("description", None),
    #
    # Columns which hold one value, or none at all: `Ref` is `mm10` for every cell that has it,
    # `Empty.Wells` is one list of wells repeated, `X.1` is the string `NA` for all 110,746 cells,
    # `X` is blank for most of them, and `not_na` is true throughout. None of them distinguishes
    # anything, so none of them is worth carrying. Nor does `cell`, which repeats the cell's own name
    # for 64,675 of them and says `NA` for the other 46,071.
    #
    # `X` is why a key names the axis and not just the property: the UMIs matrix is called `X` as
    # well, and naming that one would be `("cell", "gene", "X")`.
    ("cell", "cell"): None,
    ("cell", "Ref"): None,
    ("cell", "Empty.Wells"): None,
    ("cell", "X"): None,
    ("cell", "X.1"): None,
    ("cell", "not_na"): None,
}

cells = dp.files_daf("dafs/cells", "w", name="cells")

mc.import_cells_h5ad(cells, cells_h5ad="input/assigned_cells.h5ad", copy_data=COPY_DATA)

# The total UMIs of a cell look like data and are not: they are the sum of the UMIs already imported. An `h5ad` which
# happens to carry them is imported with them and is left alone; this one does not, so they are computed here, once,
# rather than by whichever computation needs them first.
if not cells.has_vector("cell", "total_UMIs"):
    mc.compute_vector_of_total_UMIs_per_cell(cells)

print(cells.description())

┌ Debug: Daf: FilesDaf cells path: dafs/cells
└ @ DataAxesFormats.FilesFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/DataAxesFormats/2qFvK/src/files_format.jl:409
┌ Debug: Metacells.AnnDataFormat.import_cells_h5ad! {
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:192
┌ Debug: - daf: FilesDaf cells
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:192
┌ Debug: - cells_h5ad: "input/assigned_c..." (25)
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:192
┌ Debug: - copy_data: 32 x Union{AbstractString, Tuple{AbstractString, AbstractString}, Tuple{AbstractString, AbstractString, AbstractString}} => Union{Nothing, Tuple{AbstractString, Union{Nothing, Bool, Float32, Float64, Int16, Int32, Int64, Int8, UInt16, UInt32, UInt64, UInt8, AbstractString}

┌ Debug: - insist: false
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:192
┌ Debug: skip scalar: __name__
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:556
┌ Debug: copy gene vector: lateral_gene to: is_lateral
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:645
┌ Debug: copy gene vector: rare_gene_module to: rare_module
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:645
┌ Debug: skip gene vector: full_gene_index
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:619
┌ Debug: copy gene vector: properly_sampled_gene to: is_properly_sampled
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/sh

┌ Debug: skip cell vector: batch_set_id
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:619
┌ Debug: copy cell vector: age_group_emb
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: skip cell vector: X.1
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:619
┌ Debug: copy cell vector: embryo_with_placenta_information
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: skip cell vector: full_cell_index
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:619
┌ Debug: copy cell vector: pi_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/a

┌ Debug: copy cell vector: time
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: copy cell vector: gfp_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: copy cell vector: apc_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: copy cell vector: cells_rare_gene_module to: rare_gene_module
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:645
┌ Debug: copy cell vector: embryo
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: copy cell vector: dissolved
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_

┌ Debug: copy cell vector: apc_cy7_a
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: copy cell vector: excluded_cell to: is_excluded
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:645
┌ Debug: copy cell vector: coordinates
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: skip cell vector: not_na
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:619
┌ Debug: skip cell vector: X
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:619
┌ Debug: copy cell vector: seq_batch_id to: sequencing_run
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qck

┌ Debug: copy cell vector: fsc_h
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:643
┌ Debug: skip cell vector: metacell
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:619
┌ Debug: skip cell vector: most_similar
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:619
┌ Debug: copy cell vector: QC1 to: qc1
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:645
┌ Debug: copy cell vector: Last.sequensing.date to: last_sequencing_date
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:645
┌ Debug: copy cell vector: Experiment to: experiment
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/pac

Sum 100%|████████████████████████████████████████████████| Time: 0:00:03
┌ Debug: Mean (included) UMIs per cell: 7862.727421306413
└ @ Metacells.AnalyzeCells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_cells.jl:53
┌ Debug: Metacells.AnalyzeCells.compute_vector_of_total_UMIs_per_cell! return }
└ @ Metacells.AnalyzeCells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_cells.jl:40


name: cells
type: FilesDaf
path: /net/mraid20/ifs/wisdom/tanay_lab/data/users/obk/src/metacells-sharpening-vignette/dafs/cells
mode: w
axes:
  cell: 110746 entries
  gene: 28183 entries
vectors:
  cell:
    age_group: 110,746 x Float64 (Dense)
    age_group_emb: 110,746 x Float64 (Dense)
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    average_size_bp: 110,746 x Str (Dense)
    batch: 110,746 x Str (Dense)
    comment: 110,746 x Str (Dense)
    concentration_ng_per_ul: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    delta_ct: 110,746 x Str (Dense)
    description: 110,746 x Str (Dense)
    developmental_time: 110,746 x Float64 (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x Str (Dense)
    embryo_with_placenta_information: 110,746 x Str (Dense)
    excluded_UMIs: 110,746 x UInt32 (Dense)
    experiment: 110,746 x Str (Dense)
    external_index: 110,74

## Cleaning the data


In [3]:
# How this data spells "there is no value here", which is not one way but several, sometimes several
# in the same property: `embryo` says both `NA` and nothing at all. Nothing infers these - a type
# genuinely called `NA` is possible - so each is named, and a property named here whose data happens
# to be clean is simply left alone.
#
# This has to happen before any axis is built, since building one asks of each cell whether it has a
# value: a property still saying `NA` would put `NA` on the axis, sitting among the real entries.
EMPTY_VALUES = {
    "embryo": ("NA",),
    "metacell": ("Outliers",),
    "type": ("Outliers", "Doublet"),
    "projected_type": ("(Missing)",),
    "coordinates": ("NA",),
    "source": ("NA",),
}

for property_name, empty_values in EMPTY_VALUES.items():
    dp.unify_empty_vector_values(cells, axis="cell", property=property_name, empty_values=empty_values)

# Numbers which arrived as text, because a few of their entries say `NA` and one `NA` makes a whole
# column of measurements a column of strings. Converting and unifying is one step, not two: what
# `23.5` should become is obvious, and what `NA` should become is only obvious once we are told that
# it means nothing. A number which is neither is an error rather than a silent `NaN`.
AS_NUMBERS = {
    "qc1": ("NA", np.float32),
    "qc2": ("NA", np.float32),
    "delta_ct": ("NA", np.float32),
    "concentration_ng_per_ul": ("NA", np.float32),
    # 1..32, so `0` is free to mean "none" - the convention `Daf` already uses for module indices.
    # The cells with no plate are the ones sequenced by 10x, which has no plates.
    "internal_index": ("", np.uint32),
}

for property_name, (empty_values, dtype) in AS_NUMBERS.items():
    dp.unify_empty_vector_values(
        cells, axis="cell", property=property_name, empty_values=empty_values, dtype=dtype
    )

# A sentinel which is not obviously one: the smallest 32 bit integer, which survived a cast to float
# and so is an ordinary number as far as anything reading it is concerned. Left alone, the mean of
# this property is wrong by a couple of billion rather than visibly absent.
dp.unify_empty_vector_values(
    cells, axis="cell", property="transcriptional_rank", empty_values=np.float64(-2147483648.0)
)

## Reconstructing the axes


In [4]:
# The types, and the color of each, which is what makes the graphs readable. The file decides which
# types there are and in what order they are listed - usually a meaningful order rather than an
# alphabetical one. It may name a type no cell has; a type of some cell which it does not name is an
# error, in the file or in the data. Skip this and everything still runs, uncolored.
mc.import_type_colors_csv(cells, type_colors_csv="input/type_colors.csv")

# `AnnData` has two axes, so everything else it knows is flattened onto the cells: which batch a cell
# came from, and with it every fact about that batch, repeated across its cells. Reconstructing an
# axis puts each fact where it belongs - one value per batch rather than 110,746 copies of it - and
# says so in the structure rather than in a naming convention.
#
# What is per batch, and what is merely constant within a batch by accident, is decided by looking:
# a property whose value differs between two cells of a batch is left alone. That is convenient and
# slightly dangerous, since a property which happens to be uniform is moved as readily as one which
# is uniform for a reason. These are the pipeline's own, which belong to the cells whatever their
# values happen to look like here - `is_excluded` is false for every cell of this data set, which
# says nothing about where it belongs.
KEEP_PER_CELL = {"is_excluded", "is_properly_sampled", "is_rare", "rare_gene_module", "spike_count"}

for axis in ("batch", "embryo"):
    dp.reconstruct_axis(cells, existing_axis="cell", implicit_axis=axis, skipped_properties=KEEP_PER_CELL)

# A batch was on a plate and was sequenced in a run, so those are properties of the batch now, and
# each is an axis of its own with the batch's facts divided again between them. The wet lab's record
# lands where it is read: the plate's owner and dates on the plate, the batch's concentration and QC
# on the batch, the sequencing dates on the run.
#
# The coarser axis goes first. Each plate was sequenced in one run, so a fact about a run is also
# constant within each of its plates, and reconstructing the plate first would take the run's dates
# with it - leaving the run with nothing. The reverse cannot happen: a run holds many plates, so a
# plate's own owner and dates are not constant within it.
for axis in ("sequencing_run", "plate"):
    dp.reconstruct_axis(cells, existing_axis="batch", implicit_axis=axis, skipped_properties=KEEP_PER_CELL)

# Each plate belongs to one sequencing run, but nothing has said so where a plate can be asked. It
# cannot be reconstructed: the cells sequenced by 10x have a run and no plate at all, so moving the
# run onto the plate would discard theirs. Connecting says it while leaving the batch's own run
# alone, and fails if any plate's batches disagree about which run they were in.
dp.connect_axes(cells, base_axis="batch", from_axis="plate", to_axis="sequencing_run")

print(cells.description())

┌ Debug: Metacells.AnnDataFormat.import_type_colors_csv! {
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:485
┌ Debug: - daf: FilesDaf cells
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:485
┌ Debug: - type_colors_csv: "input/type_color..." (21)
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:485
┌ Debug: - axis: "cell"
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:485
┌ Debug: - property: "type"
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:485
┌ Debug: - type_axis: "type"
└ @ Metacells.AnnDataFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/anndata_format.jl:485


name: cells
type: FilesDaf
path: /net/mraid20/ifs/wisdom/tanay_lab/data/users/obk/src/metacells-sharpening-vignette/dafs/cells
mode: w
axes:
  batch: 416 entries
  cell: 110746 entries
  embryo: 385 entries
  gene: 28183 entries
  plate: 246 entries
  sequencing_run: 195 entries
  type: 44 entries
vectors:
  batch:
    average_size_bp: 416 x Str (Dense)
    comment: 416 x Str (Dense)
    concentration_ng_per_ul: 416 x Float32 (Dense)
    delta_ct: 416 x Float32 (Dense)
    external_index: 416 x Str (Dense)
    internal_index: 416 x UInt32 (Dense)
    plate: 416 x Str (Dense)
    qc1: 416 x Float32 (Dense)
    qc2: 416 x Float32 (Dense)
    sequencing_run: 416 x Str (Dense)
  cell:
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    batch: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x Str (Dense)
    embryo_with_place

## Building the base metacells


In [5]:
# The metacells we start from, in a repository of their own resting on the cells. Everything computed
# from here on lives in such a repository, and the cells are never written to again - which is what
# lets one set of cells carry several analyses of them without copying a single UMI.
base_metacells = dp.complete_chain(
    base_daf=cells,
    new_daf=dp.files_daf("dafs/metacells.base", "w", name="metacells.base"),
    name="metacells.base",
)

# Which metacell each cell belongs to is an input here rather than something computed: it came with
# the `h5ad`, and it stays in the cells, where every analysis of them reads it. Sharpening does not
# change it - each round writes a new assignment into a repository of its own, and each starts again
# from this one.
#
# The metacells themselves, named by the values of that property. `reconstruct_axis` would also move
# to the new axis every per-cell property which happens to be constant per metacell; here it is asked
# for the axis alone, since anything per metacell is about to be computed rather than inherited.
dp.reconstruct_axis(
    base_metacells, existing_axis="cell", implicit_axis="metacell", implicit_properties=set()
)

# What the cells say about their metacells: how many cells each has, their UMIs, and the fraction of
# each gene in each of them. Then the marker genes - the ones which distinguish between metacells.
#
# Neither depends on the gene masks, which is why they are here rather than in each analysis: this
# repository is shared by all of them.
mc.prepare_metacells(base_metacells)
mc.prepare_markers(base_metacells)

print(base_metacells.description())

┌ Debug: Daf: FilesDaf metacells.base path: dafs/metacells.base
└ @ DataAxesFormats.FilesFormat ~/anaconda3/envs/metacells-sharpening/share/julia/packages/DataAxesFormats/2qFvK/src/files_format.jl:409
┌ Debug: Metacells.Pipeline.prepare_metacells! {
└ @ Metacells.Pipeline ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/pipeline.jl:69
┌ Debug: - daf: Write Chain metacells.base#2
└ @ Metacells.Pipeline ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/pipeline.jl:69
┌ Debug: - overwrite: false
└ @ Metacells.Pipeline ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/pipeline.jl:69
┌ Debug: Metacells.AnalyzeMetacells.compute_vector_of_type_per_metacell_by_cells! {
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:698
┌ Debug: - daf: Write Chain metacells.base#2
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-shar

GroupByColumns(Sum) 100%|████████████████████████████████| Time: 0:00:09
┌ Debug: Metacells.AnalyzeMetacells.compute_matrix_of_UMIs_per_gene_per_metacell! return }
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:103
┌ Debug: Metacells.AnalyzeMetacells.compute_vector_of_total_UMIs_per_metacell! {
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:129
┌ Debug: - daf: Write Chain metacells.base#2
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:129
┌ Debug: - overwrite: false
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:129
Sum 100%|████████████████████████████████████████████████| Time: 0:00:00
┌ Debug: Mean UMIs in metacell: 361093.6524263791
└ @ Metacells.Analy

┌ Debug: - overwrite: false
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:156
┌ Debug: Metacells.AnalyzeMetacells.compute_matrix_of_linear_fraction_per_gene_per_metacell! return }
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:156
┌ Debug: Metacells.AnalyzeMetacells.compute_matrix_of_log_linear_fraction_per_gene_per_metacell! {
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:203
┌ Debug: - daf: Write Chain metacells.base#2
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:203
┌ Debug: - gene_fraction_regularization: 1.0e-5
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:203
┌ D

log_linear_fraction_per_gene_per_metacell 100%|██████████| Time: 0:00:01
┌ Debug: Metacells.AnalyzeMetacells.compute_matrix_of_log_linear_fraction_per_gene_per_metacell! return }
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:203
┌ Debug: Metacells.Pipeline.prepare_metacells! return }
└ @ Metacells.Pipeline ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/pipeline.jl:69
┌ Debug: Metacells.Pipeline.prepare_markers! {
└ @ Metacells.Pipeline ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/pipeline.jl:110
┌ Debug: - daf: Write Chain metacells.base#2
└ @ Metacells.Pipeline ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/pipeline.jl:110
┌ Debug: - overwrite: false
└ @ Metacells.Pipeline ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/pipeline.jl:110
┌ Debug: Metacells.AnalyzeGenes.compute

Min 100%|████████████████████████████████████████████████| Time: 0:00:00
Max 100%|████████████████████████████████████████████████| Time: 0:00:00
┌ Debug: Marker genes: 8349
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_genes.jl:100
┌ Debug: Metacells.AnalyzeGenes.compute_vector_of_is_marker_per_gene! return }
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_genes.jl:67
┌ Debug: Metacells.AnalyzeGenes.compute_vector_of_marker_rank_per_gene! {
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_genes.jl:344
┌ Debug: - daf: Write Chain metacells.base#2
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_genes.jl:344
┌ Debug: - overwrite: false
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacel

abs_fold_per_metacell_per_marker 100%|███████████████████| Time: 0:00:00
rank_per_variable_per_observation 100%|██████████████████| Time: 0:00:01


min_rank_and_maximal_score_per_variable 100%|████████████| Time: 0:00:00
┌ Debug: Metacells.AnalyzeGenes.compute_vector_of_marker_rank_per_gene! return }
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_genes.jl:344
┌ Debug: Metacells.AnalyzeMetacells.compute_matrix_of_correlation_between_markers_per_gene_per_gene! {
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:796
┌ Debug: - daf: Write Chain metacells.base#2
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:796
┌ Debug: - overwrite: false
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/qckjR/src/analyze_metacells.jl:796
log_fraction_per_subset_metacell_per_marker_gene 100%|███| Time: 0:00:00
┌ Debug: Metacells.AnalyzeMetacells.compute_matrix_of_correlation

name: metacells.base#2
type: Write Chain
chain:
- FilesDaf cells
- FilesDaf metacells.base
scalars:
  base_daf_repository: "cells"
axes:
  batch: 416 entries
  cell: 110746 entries
  embryo: 385 entries
  gene: 28183 entries
  metacell: 2411 entries
  plate: 246 entries
  sequencing_run: 195 entries
  type: 44 entries
vectors:
  batch:
    average_size_bp: 416 x Str (Dense)
    comment: 416 x Str (Dense)
    concentration_ng_per_ul: 416 x Float32 (Dense)
    delta_ct: 416 x Float32 (Dense)
    external_index: 416 x Str (Dense)
    internal_index: 416 x UInt32 (Dense)
    plate: 416 x Str (Dense)
    qc1: 416 x Float32 (Dense)
    qc2: 416 x Float32 (Dense)
    sequencing_run: 416 x Str (Dense)
  cell:
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    batch: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x Str (Dense)
